# 03 — Exploração do arquivo fenotípico


Este notebook realiza a exploração inicial do arquivo fenotípico do ADHD-200.

### Legenda do diagnóstico (`DX`)

- `0` = controle típico
- `1` = TDAH combinado
- `2` = TDAH hiperativo-impulsivo
- `3` = TDAH desatento

Também serão verificadas as variáveis de controle de qualidade (`QC_Athena` e `QC_NIAK`), a distribuição dos participantes por site e algumas características fenotípicas.

> **Importante:** este notebook é exploratório. A definição definitiva da amostra utilizada no TCC será realizada considerando os critérios de diagnóstico, controle de qualidade e correspondência com os dados de rs-fMRI utilizados nas etapas seguintes.

In [29]:
from pathlib import Path
import sys
import pandas as pd



def encontrar_raiz_projeto():
    """Procura a pasta que contém o config.py."""
    
    candidatos = [Path.cwd(), *Path.cwd().parents]
    
    for caminho in candidatos:
        if (caminho / "config.py").exists():
            return caminho.resolve()
    
    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto "
        "TCC_TDAH."
    )


RAIZ = encontrar_raiz_projeto()

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import config

if not config.ARQ_PHENO.exists():
    raise FileNotFoundError(
        f"Arquivo de fenótipos não encontrado:\n"
        f"{config.ARQ_PHENO}"
    )

pheno = pd.read_csv(
    config.ARQ_PHENO,
    sep="\t"
)


print("ARQUIVO FENOTÍPICO — ADHD-200")


print(f"\nArquivo:")
print(f"  {config.ARQ_PHENO}")

print(f"\nDimensão:")
print(f"  {pheno.shape[0]:,} participantes × "
      f"{pheno.shape[1]:,} variáveis")

print("\nPrimeiras linhas:")
display(pheno.head())

print("\nColunas disponíveis:")
print(pheno.columns.tolist())

ARQUIVO FENOTÍPICO — ADHD-200

Arquivo:
  C:\Users\Maria Eduarda\Downloads\TCC_TDAH\data\raw\adhd200_preprocessed_phenotypics.tsv

Dimensão:
  973 participantes × 19 variáveis

Primeiras linhas:


,ScanDir ID,Site,Gender,Age,Handedness,DX,Secondary Dx,ADHD Measure,ADHD Index,Inattentive,Hyper/Impulsive,IQ Measure,Verbal IQ,Performance IQ,Full2 IQ,Full4 IQ,Med Status,QC_Athena,QC_NIAK
0,2371032,3,0.0,10.73,1,0,NaN,2,47,55,43,1.0,121.0,119.0,NaN,122.0,1,1.0,1.0
1,2026113,3,0.0,12.99,1,1,NaN,2,90,89,78,1.0,122.0,108.0,NaN,106.0,1,1.0,1.0
2,3434578,3,0.0,8.12,1,0,NaN,2,42,42,43,1.0,85.0,98.0,NaN,89.0,1,1.0,1.0
3,8628223,3,0.0,10.81,1,0,Simple phobia,2,42,49,49,1.0,85.0,86.0,NaN,97.0,1,1.0,1.0
4,1623716,3,0.0,12.65,1,1,NaN,2,87,90,90,1.0,89.0,88.0,NaN,89.0,1,1.0,1.0



Colunas disponíveis:
['ScanDir ID', 'Site', 'Gender', 'Age', 'Handedness', 'DX', 'Secondary Dx', 'ADHD Measure', 'ADHD Index', 'Inattentive', 'Hyper/Impulsive', 'IQ Measure', 'Verbal IQ', 'Performance IQ', 'Full2 IQ', 'Full4 IQ', 'Med Status', 'QC_Athena', 'QC_NIAK']


In [30]:
# Distribuição do diagnóstico e controle de qualidade



print("DISTRIBUIÇÃO DO DIAGNÓSTICO")


if "DX" not in pheno.columns:
    raise KeyError("A coluna 'DX' não foi encontrada.")

print(pheno["DX"].value_counts(dropna=False).sort_index())



print("CONTROLE DE QUALIDADE — ATHENA")


if "QC_Athena" not in pheno.columns:
    raise KeyError("A coluna 'QC_Athena' não foi encontrada.")

print(pheno["QC_Athena"].value_counts(dropna=False).sort_index())



print("CONTROLE DE QUALIDADE — NIAK")


if "QC_NIAK" not in pheno.columns:
    print("Coluna 'QC_NIAK' não encontrada no arquivo.")
else:
    print(
        pheno["QC_NIAK"]
        .value_counts(dropna=False)
        .sort_index()
    )



print("INTERPRETAÇÃO DO DX")


print("0 = Controle")
print("1 = TDAH combinado")
print("2 = TDAH hiperativo-impulsivo")
print("3 = TDAH desatento")

DISTRIBUIÇÃO DO DIAGNÓSTICO
DX
0          585
1          212
2           13
3          137
pending     26
Name: count, dtype: int64
CONTROLE DE QUALIDADE — ATHENA
QC_Athena
0.0     68
1.0    897
NaN      8
Name: count, dtype: int64
CONTROLE DE QUALIDADE — NIAK
QC_NIAK
0.0     50
1.0    915
NaN      8
Name: count, dtype: int64
INTERPRETAÇÃO DO DX
0 = Controle
1 = TDAH combinado
2 = TDAH hiperativo-impulsivo
3 = TDAH desatento


In [31]:
# Relação entre diagnóstico e QC_Athena


print("DIAGNÓSTICO × QC_ATHENA")


tabela_dx_qc = pd.crosstab(
    pheno["DX"],
    pheno["QC_Athena"],
    dropna=False
)

display(tabela_dx_qc)

DIAGNÓSTICO × QC_ATHENA


QC_Athena,0.0,1.0,NaN
DX,,,
0,35,547,3
1,18,190,4
2,1,12,0
3,13,123,1
pending,1,25,0


In [32]:
# Distribuição após QC_Athena



print("PARTICIPANTES APROVADOS NO QC_ATHENA")


pheno_qc = pheno[
    pheno["QC_Athena"] == 1
].copy()

print(
    f"Participantes aprovados no QC_Athena: "
    f"{len(pheno_qc):,}"
)

print("\nDistribuição do DX após QC_Athena:")

print(
    pheno_qc["DX"]
    .value_counts(dropna=False)
    .sort_index()
)

PARTICIPANTES APROVADOS NO QC_ATHENA
Participantes aprovados no QC_Athena: 897

Distribuição do DX após QC_Athena:
DX
0          547
1          190
2           12
3          123
pending     25
Name: count, dtype: int64


In [33]:
# Distribuição por site


print("PARTICIPANTES POR SITE")


if "Site" in pheno.columns:
    participantes_site = (
        pheno["Site"]
        .value_counts(dropna=False)
        .sort_index()
    )

    display(participantes_site)

else:
    print("A coluna 'Site' não foi encontrada.")


# Características fenotípicas



print("CARACTERÍSTICAS FENOTÍPICAS")


colunas_numericas = [
    coluna
    for coluna in [
        "Age",
        "IQ Measure",
        "Verbal IQ",
        "Performance IQ",
        "Full4 IQ"
    ]
    if coluna in pheno.columns
]

if colunas_numericas:
    display(
        pheno[colunas_numericas].describe()
    )

else:
    print(
        "Nenhuma das variáveis fenotípicas esperadas "
        "foi encontrada."
    )

PARTICIPANTES POR SITE


Site
1    245
2     26
3     94
4     73
5    263
6    113
7     98
8     61
Name: count, dtype: int64

CARACTERÍSTICAS FENOTÍPICAS


,Age,IQ Measure,Verbal IQ,Performance IQ,Full4 IQ
count,973.000000,899.000000,725.000000,725.000000,898.000000
mean,12.034872,1.273637,31.234483,25.713103,97.578842
std,3.328270,33.409364,289.861555,288.312825,122.956864
min,7.090000,-999.000000,-999.000000,-999.000000,-999.000000
25%,9.330000,2.000000,100.000000,93.000000,102.000000
50%,11.420000,2.000000,111.000000,106.000000,111.000000
75%,14.000000,3.000000,121.000000,116.000000,120.750000
max,26.310000,5.000000,158.000000,143.000000,153.000000


In [34]:
# Criar variável binária para classificação

print("=" * 60)
print("CLASSIFICAÇÃO BINÁRIA — CONTROLE × TDAH")
print("=" * 60)

# Selecionar somente participantes aprovados no QC Athena


pheno_qc = pheno[
    pheno["QC_Athena"] == 1
].copy()

print(
    f"\nParticipantes aprovados no QC_Athena: "
    f"{len(pheno_qc):,}"
)

# Remover diagnósticos pendentes


pheno_qc = pheno_qc[
    pheno_qc["DX"].isin([0, 1, 2, 3])
].copy()

print(
    f"Participantes com diagnóstico válido: "
    f"{len(pheno_qc):,}"
)

# ---------------------------------------------------------
# classificação binária
# ---------------------------------------------------------
#
# DX = 0 → Controle → target 0
# DX = 1 → TDAH     → target 1
# DX = 2 → TDAH     → target 1
# DX = 3 → TDAH     → target 1
#
# DX = pending foi excluído anteriormente.
# ---------------------------------------------------------

pheno_qc["target"] = pheno_qc["DX"].apply(
    lambda x: 0 if x == 0 else 1
)

print("\nDistribuição da variável target:")

contagem_target = (
    pheno_qc["target"]
    .value_counts()
    .sort_index()
)

display(contagem_target)

print("\nLegenda:")
print("0 = Controle")
print("1 = TDAH")

CLASSIFICAÇÃO BINÁRIA — CONTROLE × TDAH

Participantes aprovados no QC_Athena: 897
Participantes com diagnóstico válido: 0

Distribuição da variável target:


Series([], Name: count, dtype: int64)


Legenda:
0 = Controle
1 = TDAH


In [ ]:
# CÉLULA 2 — Fisher z e vetorização das matrizes
# Triângulo superior sem a diagonal: cada conexão é representada uma única vez.
iu = np.triu_indices(config.N_ROIS, k=1)
n_atributos_esperado = config.N_ROIS * (config.N_ROIS - 1) // 2

# Fisher z: o recorte evita valores infinitos para r=-1 ou r=1.
# Aplicar apenas uma vez às correlações de Pearson originais.
fc_clipped = np.clip(fc, -0.999999, 0.999999)
fc_z = np.arctanh(fc_clipped)

X = fc_z[:, iu[0], iu[1]].astype(np.float32)

if X.shape != (len(df), n_atributos_esperado):
    raise ValueError(
        f"Formato inesperado de X: {X.shape}; "
        f"esperado: {(len(df), n_atributos_esperado)}."
    )
if not np.isfinite(X).all():
    raise ValueError("X contém NaN ou valores infinitos.")

print("X:", X.shape)
print("Atributos por participante:", X.shape[1])
print("NaN:", int(np.isnan(X).sum()))
print("Infinitos:", int(np.isinf(X).sum()))
print("Transformação Fisher z aplicada: sim")

Raiz do projeto: C:\Users\Maria Eduarda\Downloads\TCC_TDAH
Configuração carregada: C:\Users\Maria Eduarda\Downloads\TCC_TDAH\config.py

Participantes carregados: 709
Formato das matrizes: (709, 190, 190)
Número de ROIs: 190
Classes encontradas: [np.int64(0), np.int64(1)]

Distribuição das classes:
classe
0    453
1    256
Name: count, dtype: int64

Número de sites: 7

Célula executada com sucesso.
A variável fc está pronta para a transformação Fisher z.


In [35]:
# Resumo da exploração


print("RESUMO — ADHD-200")


print(f"\nParticipantes no arquivo fenotípico: {len(pheno):,}")

print(
    f"Participantes aprovados no QC_Athena: "
    f"{len(pheno_qc):,}"
)

print("\nDistribuição final após QC:")

contagem_target = (
    pheno_qc["target"]
    .value_counts()
    .sort_index()
)

for classe, quantidade in contagem_target.items():

    nome = "Controle" if classe == 0 else "TDAH"

    percentual = (
        quantidade / len(pheno_qc) * 100
    )

    print(
        f"  {nome}: {quantidade:,} "
        f"({percentual:.2f}%)"
    )


print("✓ EXPLORAÇÃO CONCLUÍDA")



RESUMO — ADHD-200

Participantes no arquivo fenotípico: 973
Participantes aprovados no QC_Athena: 0

Distribuição final após QC:
✓ EXPLORAÇÃO CONCLUÍDA
